# P4 · DistilBERT CVE-severity fine-tune (Colab free GPU)

Runtime → Change runtime type → **T4 GPU**. Runs the same scripts and the same stratified split as the TF-IDF baseline, so the macro-F1 numbers are directly comparable.

1. Set `REPO` to your fork's URL (or upload the repo as a zip).
2. Run all. ~10 min total (NVD download ~2 min, training ~5-8 min).
3. Download `distilbert-cve-severity.zip` and unzip into `ml/models/` locally so the triage agents use it.

In [ ]:
REPO = 'https://github.com/<you>/policycraft.git'  # <- change me
!git clone -q $REPO policycraft
%cd policycraft
!pip -q install -r ml/requirements.txt
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# Optional: NVD API key makes the download ~8x faster (free: https://nvd.nist.gov/developers/request-an-api-key)
# import os; os.environ['NVD_API_KEY'] = '...'
![ -f ml/data/nvd_cves.csv ] && echo "Using committed dataset (same split as the baseline)" || python -m ml.fetch_nvd --per-class 1500 --out ml/data/nvd_cves.csv

In [ ]:
!python -m ml.baseline --data ml/data/nvd_cves.csv

In [ ]:
!python -m ml.train_distilbert --data ml/data/nvd_cves.csv --epochs 3 --batch 32 --max-len 256

In [ ]:
!python -m ml.compare

In [ ]:
!cd ml/models && rm -rf distilbert-cve-severity/checkpoints && zip -qr ../../distilbert-cve-severity.zip distilbert-cve-severity
from google.colab import files
files.download('distilbert-cve-severity.zip')
files.download('ml/results/distilbert.json')